In [1]:
#add main path
import sys
sys.path.append('/Users/lmwgsrwny/Documents/KWS-on-Mamba/')

import torch
from torch.utils.data import DataLoader
import torch.optim as optim
import torch.nn as nn
from torch_optimizer import Lookahead
from src.data.config import dataset, data_loader, model as model_config, optimizer as optimizer_config, scheduler as scheduler_config, training

# Import custom modules
from src.models.model import KeywordSpottingModel_with_cls
from src.data.data_loader import load_speech_commands_dataset, TFDatasetAdapter, load_bg_noise_dataset
from src.utils.utils import set_memory_GB, print_model_size, log_to_file, plot_learning_curves
from src.utils.augmentations import add_time_shift_and_align, add_silence
from src.utils.train_utils import trainig_loop


/Users/lmwgsrwny/Documents/KWS-on-Mamba/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/lmwgsrwny/Documents/KWS-on-Mamba/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# Load datasets
train_ds, val_ds, test_ds, silence_ds , info = load_speech_commands_dataset(reduced=True)
# bg_noise_ds = load_bg_noise_dataset()

In [3]:

# Initialize datasets with configurations
pytorch_test_dataset = TFDatasetAdapter(test_ds, None, **dataset, augmentation=[lambda x: add_time_shift_and_align(x)])
#

2025-01-01 18:27:57.444386: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:376] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608
2025-01-01 18:27:57.637481: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [4]:

# # Create DataLoaders
# train_loader = DataLoader(pytorch_train_dataset, **data_loader, shuffle=True)
# val_loader = DataLoader(pytorch_val_dataset, **data_loader, shuffle=False)


In [5]:
device = torch.device("mps" if torch.mps.is_available() else "cpu")
print(f"Device: {device}")

Device: mps


In [6]:

# Initialize model
model = KeywordSpottingModel_with_cls(**model_config)

# Loss function
criterion = nn.CrossEntropyLoss().to(device)

# Optimizer
base_optimizer = optim.Adam(model.parameters(), lr=optimizer_config['lr'], weight_decay=optimizer_config['weight_decay'])
optimizer = Lookahead(base_optimizer, **optimizer_config['lookahead'])

# Scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, **scheduler_config['reduce_lr_on_plateau'])


In [126]:
# load test data
pytorch_test_dataset = TFDatasetAdapter(test_ds, None, **dataset, augmentation=None)
test_loader = DataLoader(pytorch_test_dataset, **data_loader, shuffle=False)


In [8]:
model.load_state_dict(torch.load('/Users/lmwgsrwny/Documents/KWS-on-Mamba/best_model.pth',map_location=torch.device(device)))


/var/folders/f5/gsx5dxy538z0xyk3rpdwmjqr0000gn/T/ipykernel_56189/965035254.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('/Users/lmwgs

<All keys matched successfully>

In [9]:
model = model.to(device)

In [10]:

# Evaluate the model on the test set
accuracy = 0
total = 0

model.eval()

with torch.no_grad():
    for audio, labels in test_loader:
        audio, labels = audio.to(device), labels.to(device)
        outputs = model(audio)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        accuracy += (predicted == labels).sum().item()
test_accuracy = 100 * accuracy / total
print(f'Test Accuracy: {test_accuracy}%')



Test Accuracy: 93.91261659302896%


# Export to ONNX:

In [11]:
for audio, labels in test_loader:
    dummy_input = audio.to(device)
    # take first item in batch
    break
        
print(dummy_input.shape)

torch.Size([26, 69, 135])


In [12]:
torch.onnx.export(model, dummy_input, "model.onnx", export_params=True, opset_version=16)

/Users/lmwgsrwny/Documents/KWS-on-Mamba/mambaPy/mamba.py/mambapy/pscan.py:18: TracerWarning: Converting a tensor to a Python float might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  return 2 ** math.ceil(math.log2(len))
/Users/lmwgsrwny/Documents/KWS-on-Mamba/mambaPy/mamba.py/mambapy/pscan.py:168: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if L == npo2(L):
/Users/lmwgsrwny/Documents/KWS-on-Mamba/mambaPy/mamba.py/mambapy/pscan.py:49: TracerWarning: Converting a tensor to a Python float might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a con

In [116]:
# Prepare calibration data
calibration_data = []

for audio, _ in test_loader:
    audio_np = audio.numpy()  # Convert PyTorch tensors to NumPy arrays
    # Ensure the feature dimensions are [69, 135]
    reshaped_audio = audio_np.reshape(-1, 69, 135)
    calibration_data.extend(reshaped_audio)

    # Stop after collecting enough samples for calibration
    if len(calibration_data) >= 1000:
        break

In [117]:
from onnxruntime.quantization import CalibrationDataReader

class MyCalibrationDataReader(CalibrationDataReader):
    def __init__(self, calibration_data, input_name):
        self.data = calibration_data
        self.index = 0
        self.input_name = input_name

    def get_next(self):
        if self.index < len(self.data):
            batch = self.data[self.index:self.index + 1]  # ONNX expects batched inputs
            self.index += 1
            return {self.input_name: batch}
        return None

In [122]:
#quantize model
from onnxruntime.quantization import quantize_dynamic, QuantType

# quantized_model = quantize_dynamic("model.onnx", "quantized_model.onnx",weight_type=QuantType.QUInt8)
quantized_model = quantize_static("model.onnx", "quantized_model.onnx",weight_type=QuantType.QInt8,calibration_data_reader=MyCalibrationDataReader(calibration_data, input_name))

In [131]:
import onnx
onnx_model = onnx.load("quantized_model.onnx")
onnx.checker.check_model(onnx_model)
print("ONNX model is valid!")

ONNX model is valid!


# Quantize the Model:

In [132]:
import onnxruntime as ort

session = ort.InferenceSession("model.onnx")
input_name = session.get_inputs()[0].name

In [133]:
import onnxruntime as ort
import numpy as np

# Load the ONNX quantized model
session = ort.InferenceSession("quantized_model.onnx")

# Get model input and output names
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

# Initialize accuracy tracking
total = 0
correct = 0

# Iterate over test data
for audio, labels in test_loader:
    # Convert audio and labels to NumPy arrays
    audio = audio.numpy()
    labels = labels.numpy()


    outputs = session.run([output_name], {input_name: audio})

    # Convert outputs to class predictions
    predicted = np.argmax(outputs[0], axis=1)

    # Update accuracy metrics
    total += labels.shape[0]
    correct += (predicted == labels).sum()

# Calculate test accuracy
test_accuracy = 100 * correct / total
print(f"Test Accuracy: {test_accuracy:.2f}%")

Test Accuracy: 29.70%
